<a href="https://colab.research.google.com/github/prinny2/leadbellus-app/blob/main/notebook_nlp_leadbellus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 LeadBellus NLP — teste rápido

Classifica mensagens de leads (WhatsApp/Instagram) em **PT-BR** por **intenção** (zero-shot) e **sentimento**.

> Para acelerar: *Ambiente de execução → Alterar o tipo de ambiente → GPU* (opcional; a CPU também roda).

In [ ]:
!pip -q install -U transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 84.9 MB/s eta 0:00:00


## 1) Intenção da mensagem (zero-shot, sem treino)

In [ ]:
from transformers import pipeline

# Modelo multilíngue de zero-shot (entende português)
classificador = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli',
)

# Rotulos de intencao - ajuste ao seu funil (solar, estetica, etc.)
INTENCOES = [
    'pergunta de preço',
    'achou caro ou objeção',
    'quer agendar',
    'dúvida sobre o procedimento',
    'demonstra interesse',
    'pediu desconto',
    'vai pensar ou sem interesse',
]

TEMPLATE = 'Esta mensagem do cliente é sobre {}.'

def classificar(texto):
    r = classificador(texto, INTENCOES, hypothesis_template=TEMPLATE, multi_label=False)
    return r['labels'][0], round(float(r['scores'][0]), 3)

print('Modelo de intenção carregado ✅')

In [ ]:
mensagens = [
    'Oi, quanto tá o preenchimento labial?',
    'Achei meio caro, na outra clínica é mais barato',
    'Consigo um horário pra sábado de manhã?',
    'Dói muito? Tenho medo de ficar artificial',
    'Vi seu Instagram, quero energia solar pra minha casa',
    'Depois eu te chamo, vou pensar',
]

for m in mensagens:
    intent, score = classificar(m)
    print(f'[{intent}  ({score})]  {m}')

## 2) Sentimento (1 a 5 estrelas, multilíngue)

In [ ]:
sentimento = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
)

for m in mensagens:
    s = sentimento(m)[0]
    label, sc = s['label'], round(s['score'], 2)
    print(f'[{label}  ({sc})]  {m}')

## 3) Teste a sua própria mensagem

In [ ]:
texto = 'Oi, vi o anúncio de energia solar. Quanto fica pra uma casa com conta de 600 reais?'  # << troque aqui

intent, score = classificar(texto)
s = sentimento(texto)[0]
label, sc = s['label'], round(s['score'], 2)
print('Mensagem  :', texto)
print('Intenção  :', intent, f'({score})')
print('Sentimento:', label, f'({sc})')

## Próximos passos
- Ajustar a lista `INTENCOES` para o funil da LeadBellus (solar) ou da estética.
- Conectar a saída a uma automação (ex.: **Zapier** roteia o lead conforme a intenção).
- Para mais precisão, dá pra *fine-tunar* um modelo PT-BR (ex.: BERTimbau) com seus exemplos rotulados.